# 🧬 Pangenome Analysis Pipeline 
## Part 1: Data Acquisition & Preparation


**Target Organism:** *[Acinetobacter baumannii]*  

---

### 🎯 Objective
The goal of this notebook is to fetch, extract, and organize the raw genomic data (`FASTA` files) for our selected 50 high-quality complete genomes from the **NCBI RefSeq** database.

### 🗺️ Workflow in this Notebook:
1. Download compressed genomes using `ncbi-datasets-cli` based on our curated accession list.
2. Unzip the downloaded archive.
3. Extract all `.fna` (FASTA) files.
4. Consolidate them into a single clean working directory (`my_genomes/`).

Let's get started! 🚀

In [ ]:
%%bash

# 1. Download the genomes using the accession list
echo "⏳ Downloading 50 genomes from NCBI..."
datasets download genome accession --inputfile accessions_to_download.txt --include genome

# 2. Unzip the downloaded archive silently
echo "📦 Unzipping downloaded files..."
unzip -q ncbi_dataset.zip

# 3. Create a dedicated directory for our raw FASTA files
mkdir -p my_genomes

# 4. Find and copy all .fna (FASTA) files to our new directory
echo "🚚 Moving FASTA files to 'my_genomes' directory..."
find ncbi_dataset/data/ -name "*.fna" -exec cp {} my_genomes/ \;

# 5. Verify the number of downloaded genomes
echo "✅ Done! Total FASTA files successfully extracted:"
ls -1 my_genomes/*.fna | wc -l

### 🕵️ Metadata Quality Check (Avoiding Sampling Bias)
Before proceeding to genome annotation, we need to ensure our 50 randomly selected genomes are diverse. 
We will parse the `assembly_data_report.jsonl` file generated by NCBI to inspect the **Geographic Locations** and **Isolation Sources** of our genomes.

A good pangenome analysis should ideally include isolates from various regions and clinical/environmental backgrounds.

In [ ]:
import json
from collections import Counter

countries = []
sources = []

file_path = 'ncbi_dataset/data/assembly_data_report.jsonl'

try:
    with open(file_path, 'r') as f:
        for line in f:
            data = json.loads(line)
            # Safely navigate the JSON structure
            biosample = data.get('assemblyInfo', {}).get('biosample', {})
            attributes = biosample.get('attributes', [])
            
            for attr in attributes:
                if attr.get('name') == 'geo_loc_name':
                    countries.append(attr.get('value'))
                elif attr.get('name') == 'isolation_source':
                    sources.append(attr.get('value'))
                    
    print("🌍 Top Geographic Locations:")
    if countries:
        for country, count in Counter(countries).most_common(5):
            print(f" - {country}: {count}")
    else:
        print(" - No geographic data found.")
        
    print("\n🧫 Top Isolation Sources:")
    if sources:
        for source, count in Counter(sources).most_common(5):
            print(f" - {source}: {count}")
    else:
        print(" - No isolation source data found.")
        
except FileNotFoundError:
    print(f"Error: Could not find {file_path}. Make sure the dataset was extracted correctly.")
except Exception as e:
    print(f"An error occurred while parsing metadata: {e}")

### 🎉 Conclusion 
We have successfully completed the data acquisition phase:
- **Total genomes acquired:** 50 high-quality, complete genomes.
- **Data Integrity:** Raw `.fna` files are neatly consolidated in the `my_genomes/` directory.
- **Metadata Quality:** Verified the absence of extreme sampling bias, confirming acceptable geographic and clinical diversity among the isolates.

**Next Steps:** Proceed to `02_Genome_Annotation.ipynb` to annotate these raw sequences using **Prokka** and generate standardized `.gff3` files for downstream pangenome analysis.